# WP1.7 - Voltage regulator (LDC)

This notebook demonstrates the operation of the multiconductor power flow with line drop control through transformer tap changers. The controller is generally designed for individual phase control, as opposed to gang-control. The proof is based on the tests defined in the WP1.7 specification document. 

## Remarks about this controller 

### Operation modes 
The controller is generally designed and tested for 3-phased 2-winding transformers with tap at the secondary (low-voltage) side. It supports two operation modes: "locked_forward" and "locked_reverse". In locked_forward mode, the control is only active when there is a power flow from the transformer primary side to the secondary side. Otherwise, the tap settings stay neutral in that case. In locked_reverse mode, the same is the case for reverse power flow direction. There is no "gang controlled" operation mode. 

### Controller convergence 
The goal of the control action is to stabilize the voltage at a remote bus in the low voltage side grid within a given bandwidth. This voltage is estimated by measuring the current through the transformer and using the grid impedance from the transformer secondary busbar to the remote voltage-controlled bus. The latter grid impedance is to be selected by the user, and is valid for all grid phases. This method is however generally limited because the voltage at the controlled bus also depends on the configuration of loads connected along the path to the transformer. Hence, in complex networks, the controller performance will much depend on the selection of the impedance. 

Apart from this, there are two notable reasons why the controller might be unable to stabilize the voltage within the bandwidth:
(1) Limitation of the tap setting. If this is the case, the controller will drive the tap to the minimum or maximum position. 
(2) Too narrow bandwidth selection. If this is the case (especially if the target voltage bandwith is lower than the effect of one tap position change), the controller might start to oscillate between two tap settings, and thus not converge. 

All in all, care needs to be taken for both impedance (r_ldc_v, x_ldc_v) and bandwidth controller parameters to avoid poor control performance and non-convergence. 

### Phase-individual control
In order to define impedances and/or bandwidths individually per phase, it is possible to create one controller per phase and use it with 1-phased transformers.


## Basic imports

Here, the basic needed functions are imported and the test grids are loaded. 

In [1]:
import multiconductor as mc
import multiconductor.test.testing_toolbox
from multiconductor.pycci.std_types import create_std_type
from multiconductor.control.line_drop_control import add_line_drop_control, check_ldc_stable_state
from pandapower.control import run_control
import warnings
import pandas as pd 
from math import sqrt
import numpy as np
import copy 

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option('display.float_format', '{:.6f}'.format) 


In [2]:
grids = mc.test.testing_toolbox.load_all_test_grids("wp1_7_ldc")

In [3]:
l = []
for name, net in grids.items():
    net = copy.deepcopy(net)
    mc.run_pf(net, run_control=True)
    c = multiconductor.test.testing_toolbox.compare_to_dss(net)
    l.append(c)
    print('.', end='')
print()    
pd.DataFrame(l, index=grids.keys()).sort_index(key=lambda x: x.str.len())

....................


,max_vm_pu_diff,max_va_degree_diff,max_i_from_ka_diff,max_i_to_ka_diff,max_p_from_mw_diff,max_p_to_mw_diff,max_q_from_mvar_diff,max_q_to_mvar_diff
twenty_bus_system_controlling_bus2_locked_reverse_LDC.pkl,0.000001,0.000037,0.000001,0.000001,0.000000,0.000000,0.000000,0.000000
twenty_bus_system_controlling_bus1_locked_reverse_LDC.pkl,0.000001,0.000037,0.000001,0.000001,0.000000,0.000000,0.000000,0.000000
twenty_bus_system_controlling_bus1_locked_forward_LDC.pkl,0.000000,0.000013,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
twenty_bus_system_controlling_bus2_locked_forward_LDC.pkl,0.000000,0.000019,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
four_bus_system_controlling_bus3_with_symmetric_sgen_locked_reverse_LDC.pkl,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
four_bus_system_controlling_bus2_with_symmetric_load_locked_forward_LDC.pkl,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
four_bus_system_controlling_bus2_with_symmetric_load_locked_reverse_LDC.pkl,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
four_bus_system_controlling_bus2_with_symmetric_sgen_locked_forward_LDC.pkl,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
four_bus_system_controlling_bus2_with_symmetric_sgen_locked_reverse_LDC.pkl,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
four_bus_system_controlling_bus3_with_symmetric_sgen_locked_forward_LDC.pkl,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## Controller validation function

In order to validate the controller operation, the function "check_ldc_stable_state" was designed. It checks if the controller was successful in stabilizing the voltage at the controlled bus within the given voltage bandwidth, or otherwise would have used maximum or minimum tap changer settings. The latter is an allowed state for cases in which the controller is unable to maintain the voltage bandwidth given the limitations of the tap changer. Cases where the controller is deactivated due to its operation mode ("locked_forward" or "locked_reverse") are also detected by the check function. 
The following code demonstrates this function. 

In [4]:
net = copy.deepcopy(grids["four_bus_system_controlling_bus2_with_symmetric_load_locked_forward_LDC.pkl"])

print(f"grid contains {len(net.controller)} controller(s):")
for o in net.controller.object:
    print("    ", o)

print()
print("--- Run without controllers ---")
mc.run_pf(net, run_control=False)
passed = check_ldc_stable_state(net, controlled_bus_index=2, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_forward')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))

print()
print("--- Run with controllers and re-check ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=2, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_forward')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))


grid contains 1 controller(s):
     LineDropControlExtended of trafo1ph 0

--- Run without controllers ---
Check passed: False
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9605435355437053, 0.9605435355437051, 0.9605435355437055]

--- Run with controllers and re-check ---
Check passed: True
Tap settings: [3, 3, 3]
Voltages at controlled bus: [0.9803593306655018, 0.9803593306655018, 0.980359330665502]


As can be seen the controller is able to stabilize the voltage within the bandwidth in this case. 
In order to create a volt/var controller one can use the function add_line_drop_control.

In [5]:
net = copy.deepcopy(grids["four_bus_system_controlling_bus2_with_symmetric_load_locked_forward_LDC.pkl"])
# delete existing controllers
net.controller.drop(net.controller.index, inplace=True)

# Setup controller parameters and add controller
# Note: controlled bus nominal voltage is 16 kV
ct_primary_rating_a = 5
pt_ratio = 16000/120
r_ldc_v = 0.24 * ct_primary_rating_a/pt_ratio # Value chosen with respect to controlled bus 
x_ldc_v = 0.84 * ct_primary_rating_a/pt_ratio # Value chosen with respect to controlled bus
add_line_drop_control(net, trafo_top_level_index=0, mode="locked_forward", 
                      v_set_secondary_v=120.0, bandwidth_secondary_v=2.4, 
                      pt_ratio = pt_ratio, ct_primary_rating_a = ct_primary_rating_a, r_ldc_v=r_ldc_v, x_ldc_v=x_ldc_v)

print(f"grid contains {len(net.controller)} controller(s):")
for o in net.controller.object:
    print("    ", o)

mc.run_pf(net, run_control=True)
check_ldc_stable_state(net, controlled_bus_index=2, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_forward')
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))

INFO:pandapower.control.util.auxiliary:Creating controller 0 of type <class 'multiconductor.control.line_drop_control.LineDropControlExtended'> 


grid contains 1 controller(s):
     LineDropControlExtended of trafo1ph 0
Tap settings: [3, 3, 3]
Voltages at controlled bus: [0.9803593306655018, 0.9803593306655018, 0.980359330665502]


# Tests with 4-bus topologies 


## Test 1: Symmetrical load at Bus3, LDC controlling Bus2 in locked_forward mode
Controller to stabilize controlled bus voltage within voltage band of +-2% p.U.

In [6]:
net = grids["four_bus_system_controlling_bus2_with_symmetric_load_locked_forward_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=2, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_forward')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9605435355437053, 0.9605435355437051, 0.9605435355437055]

--- Post control results ---
Check passed: True
Tap settings: [3, 3, 3]
Voltages at controlled bus: [0.9803593306655018, 0.9803593306655018, 0.980359330665502]
Max overall vm_pu difference: 1.7604504742863014e-08
Max overall va_degree difference: 2.5087544397450756e-07
Max overall i_from_ka difference: 3.558922045954205e-10
Max overall i_to_ka difference: 3.558922045954205e-10
Max overall p_from_mw difference: 1.517624159674824e-08
Max overall p_to_mw difference: 1.045899367113634e-08
Max overall q_from_mvar difference: 3.502168999247601e-08
Max overall q_to_mvar difference: 1.850593456254046e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000   -179.995668       0.000000
      1      1.0000

## Test 2: Symmetrical load at Bus3, LDC controlling Bus2 in locked_reverse mode
Controller to stay inactive due to operation mode

In [7]:
net = grids["four_bus_system_controlling_bus2_with_symmetric_load_locked_reverse_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=2, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_reverse')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9605435355437053, 0.9605435355437051, 0.9605435355437055]

--- Post control results ---
Check passed: True
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9605435355437053, 0.9605435355437051, 0.9605435355437055]
Max overall vm_pu difference: 1.7321657885815966e-08
Max overall va_degree difference: 2.5742054088340183e-07
Max overall i_from_ka difference: 7.175882110743714e-10
Max overall i_to_ka difference: 7.175882110743714e-10
Max overall p_from_mw difference: 1.1832654855936653e-08
Max overall p_to_mw difference: 7.024361536878132e-09
Max overall q_from_mvar difference: 3.457946640139653e-08
Max overall q_to_mvar difference: 1.7744239633188386e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000   -111.797077       0.000000
      1      1.0

## Test 3: Symmetrical load at Bus3, LDC controlling Bus3 in locked_forward mode
Controller to stabilize controlled bus voltage within voltage band of +-2% p.U.

In [8]:
net = grids["four_bus_system_controlling_bus3_with_symmetric_load_locked_forward_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=3, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_forward')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9244847131999759, 0.924484713199976, 0.924484713199976]

--- Post control results ---
Check passed: True
Tap settings: [9, 9, 9]
Voltages at controlled bus: [0.9862104292202107, 0.9862104292201892, 0.9862104292201895]
Max overall vm_pu difference: 1.8036452886782683e-08
Max overall va_degree difference: 2.2930788912134403e-07
Max overall i_from_ka difference: 7.912915878094395e-10
Max overall i_to_ka difference: 7.912915878094395e-10
Max overall p_from_mw difference: 1.2200440657750278e-08
Max overall p_to_mw difference: 8.015329733268572e-09
Max overall q_from_mvar difference: 3.0465379108335355e-08
Max overall q_to_mvar difference: 1.5813359732774757e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000     91.346956       0.000000
      1      1.00

## Test 4: Symmetrical load at Bus3, LDC controlling Bus3 in locked_reverse mode
Controller to stay inactive due to operation mode

In [9]:
net = grids["four_bus_system_controlling_bus3_with_symmetric_load_locked_reverse_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=3, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_reverse')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9244847131999759, 0.924484713199976, 0.924484713199976]

--- Post control results ---
Check passed: True
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9244847131999759, 0.924484713199976, 0.924484713199976]
Max overall vm_pu difference: 1.7321657885815966e-08
Max overall va_degree difference: 2.5742054088340183e-07
Max overall i_from_ka difference: 7.175882110743714e-10
Max overall i_to_ka difference: 7.175882110743714e-10
Max overall p_from_mw difference: 1.1832654855936653e-08
Max overall p_to_mw difference: 7.024361536878132e-09
Max overall q_from_mvar difference: 3.457946640139653e-08
Max overall q_to_mvar difference: 1.7744239633188386e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000   -111.797077       0.000000
      1      1.00000

## Test 5: Asymmetrical load at Bus3, LDC controlling Bus2 in locked_forward mode
Controller to stabilize controlled bus voltage within voltage band of +-2% p.U.

In [10]:
net = grids["four_bus_system_controlling_bus2_with_asymmetric_load_locked_forward_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=2, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_forward')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9956702740310412, 0.9700721761502836, 0.982956879295032]

--- Post control results ---
Check passed: True
Tap settings: [0, 2, 0]
Voltages at controlled bus: [0.9956403365609712, 0.9832523538563727, 0.9829206808170459]
Max overall vm_pu difference: 1.7338914637399228e-08
Max overall va_degree difference: 2.0661856581227767e-07
Max overall i_from_ka difference: 4.0765900655159726e-09
Max overall i_to_ka difference: 4.0765900655159726e-09
Max overall p_from_mw difference: 3.234159962062222e-08
Max overall p_to_mw difference: 2.908770113663195e-08
Max overall q_from_mvar difference: 3.7267746638702803e-08
Max overall q_to_mvar difference: 2.7599041690917758e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000    155.298934     155.298933
      1      1.

## Test 6: Asymmetrical load at Bus3, LDC controlling Bus2 in locked_reverse mode
Controller to stay inactive due to operation mode

In [11]:
net = grids["four_bus_system_controlling_bus2_with_asymmetric_load_locked_reverse_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=2, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_reverse')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9956702740310412, 0.9700721761502836, 0.982956879295032]

--- Post control results ---
Check passed: True
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9956702740310412, 0.9700721761502836, 0.982956879295032]
Max overall vm_pu difference: 1.7192386292386175e-08
Max overall va_degree difference: 2.1485637091700482e-07
Max overall i_from_ka difference: 3.693710559637964e-09
Max overall i_to_ka difference: 3.693710559637964e-09
Max overall p_from_mw difference: 3.271919712943827e-08
Max overall p_to_mw difference: 2.933353471235023e-08
Max overall q_from_mvar difference: 3.8079732145313017e-08
Max overall q_to_mvar difference: 2.8039164073900835e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000    155.505623       0.000000
      1      1.000

## Test 7: Asymmetrical load at Bus3, LDC controlling Bus3 in locked_forward mode
Controller to stabilize controlled bus voltage within voltage band of +-2% p.U.

In [12]:
net = grids["four_bus_system_controlling_bus3_with_asymmetric_load_locked_forward_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=3, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_forward')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9916963556832187, 0.9419849100525869, 0.9694697563686662]

--- Post control results ---
Check passed: True
Tap settings: [0, 6, 2]
Voltages at controlled bus: [0.991521596376154, 0.983033954666796, 0.982673213118502]
Max overall vm_pu difference: 1.7633918325543618e-08
Max overall va_degree difference: 1.939661444794183e-07
Max overall i_from_ka difference: 4.815803311686295e-09
Max overall i_to_ka difference: 4.815803311686295e-09
Max overall p_from_mw difference: 3.2518933323899546e-08
Max overall p_to_mw difference: 2.949784949635159e-08
Max overall q_from_mvar difference: 3.754879884798612e-08
Max overall q_to_mvar difference: 2.891043221486811e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000    155.151348     155.151345
      1      1.000000

## Test 8: Asymmetrical load at Bus3, LDC controlling Bus3 in locked_reverse mode
Controller to stay inactive due to operation mode

In [13]:
net = grids["four_bus_system_controlling_bus3_with_asymmetric_load_locked_reverse_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=3, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_reverse')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9916963556832187, 0.9419849100525869, 0.9694697563686662]

--- Post control results ---
Check passed: True
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9916963556832187, 0.9419849100525869, 0.9694697563686662]
Max overall vm_pu difference: 1.7192386292386175e-08
Max overall va_degree difference: 2.1485637091700482e-07
Max overall i_from_ka difference: 3.693710559637964e-09
Max overall i_to_ka difference: 3.693710559637964e-09
Max overall p_from_mw difference: 3.271919712943827e-08
Max overall p_to_mw difference: 2.933353471235023e-08
Max overall q_from_mvar difference: 3.8079732145313017e-08
Max overall q_to_mvar difference: 2.8039164073900835e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000    155.505623       0.000000
      1      1.0

## Test 9: Symmetrical sgen at Bus3, LDC controlling Bus2 in locked_forward mode
Controller to stay inactive due to operation mode

In [14]:
net = grids["four_bus_system_controlling_bus2_with_symmetric_sgen_locked_forward_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=2, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_forward')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.0258986632532117, 1.0258986632532112, 1.0258986632532117]

--- Post control results ---
Check passed: True
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.0258986632532117, 1.0258986632532112, 1.0258986632532117]
Max overall vm_pu difference: 1.5622275695292842e-08
Max overall va_degree difference: 1.7344541447528172e-07
Max overall i_from_ka difference: 3.298440409693626e-09
Max overall i_to_ka difference: 3.298440409693626e-09
Max overall p_from_mw difference: 2.698417489455096e-09
Max overall p_to_mw difference: 3.2904479141393495e-09
Max overall q_from_mvar difference: 1.356914114403196e-08
Max overall q_to_mvar difference: 6.782539196947823e-09

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000   -143.125770       0.000000
      1      1.00

## Test 10: Symmetrical sgen at Bus3, LDC controlling Bus2 in locked_reverse mode
Controller to stabilize controlled bus voltage within voltage band of +-2% p.U.

In [15]:
net = grids["four_bus_system_controlling_bus2_with_symmetric_sgen_locked_reverse_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=2, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_reverse')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.0258986632532117, 1.0258986632532112, 1.0258986632532117]

--- Post control results ---
Check passed: True
Tap settings: [-1, -1, -1]
Voltages at controlled bus: [1.0197464988399274, 1.019746498839927, 1.019746498839927]
Max overall vm_pu difference: 1.55247100730449e-08
Max overall va_degree difference: 1.7685364639419277e-07
Max overall i_from_ka difference: 3.5602153447555906e-09
Max overall i_to_ka difference: 3.5602153447555906e-09
Max overall p_from_mw difference: 4.0507348586515945e-09
Max overall p_to_mw difference: 1.8492940512260247e-09
Max overall q_from_mvar difference: 2.0077833395859557e-08
Max overall q_to_mvar difference: 1.016681050458601e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000    -89.995668       0.000000
      1      1

## Test 11: Symmetrical sgen at Bus3, LDC controlling Bus3 in locked_forward mode
Controller to stay inactive due to operation mode

In [16]:
net = grids["four_bus_system_controlling_bus3_with_symmetric_sgen_locked_forward_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=3, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_forward')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.054582666603267, 1.0545826666032665, 1.054582666603267]

--- Post control results ---
Check passed: True
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.054582666603267, 1.0545826666032665, 1.054582666603267]
Max overall vm_pu difference: 1.5622275695292842e-08
Max overall va_degree difference: 1.7344541447528172e-07
Max overall i_from_ka difference: 3.298440409693626e-09
Max overall i_to_ka difference: 3.298440409693626e-09
Max overall p_from_mw difference: 2.698417489455096e-09
Max overall p_to_mw difference: 3.2904479141393495e-09
Max overall q_from_mvar difference: 1.356914114403196e-08
Max overall q_to_mvar difference: 6.782539196947823e-09

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000   -143.125770       0.000000
      1      1.000000

## Test 12: Symmetrical sgen at Bus3, LDC controlling Bus3 in locked_reverse mode
Controller to stabilize controlled bus voltage within voltage band of +-2% p.U.

In [17]:
net = grids["four_bus_system_controlling_bus3_with_symmetric_sgen_locked_reverse_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=3, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_reverse')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.054582666603267, 1.0545826666032665, 1.054582666603267]

--- Post control results ---
Check passed: True
Tap settings: [-6, -6, -6]
Voltages at controlled bus: [1.0185840849089325, 1.0185840849089323, 1.018584084908932]
Max overall vm_pu difference: 1.5036845435467683e-08
Max overall va_degree difference: 1.8455087058555364e-07
Max overall i_from_ka difference: 4.032518319263545e-09
Max overall i_to_ka difference: 4.032518319263545e-09
Max overall p_from_mw difference: 7.383958333662122e-09
Max overall p_to_mw difference: 4.499675299030059e-09
Max overall q_from_mvar difference: 2.0644271958580873e-08
Max overall q_to_mvar difference: 1.054582310366925e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000      0.004332       0.000000
      1      1.00

## Test 13: Asymmetrical sgen at Bus3, LDC controlling Bus2 in locked_forward mode
Controller to stay inactive due to operation mode

In [18]:
net = grids["four_bus_system_controlling_bus2_with_asymmetric_sgen_locked_forward_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=2, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_forward')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.0205298737331732, 1.0289850336961535, 1.0252874436236765]

--- Post control results ---
Check passed: True
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.0205298737331732, 1.0289850336961535, 1.0252874436236765]
Max overall vm_pu difference: 4.035416608511033e-07
Max overall va_degree difference: 0.0007916724957652832
Max overall i_from_ka difference: 2.521123406817871e-06
Max overall i_to_ka difference: 2.521123406817871e-06
Max overall p_from_mw difference: 2.4955604344789606e-05
Max overall p_to_mw difference: 2.577318307306342e-05
Max overall q_from_mvar difference: 1.1478913698681481e-05
Max overall q_to_mvar difference: 1.0240014758577942e-05

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000     24.500211       0.000000
      1      1.00

## Test 14: Asymmetrical sgen at Bus3, LDC controlling Bus2 in locked_reverse mode
Controller to stabilize controlled bus voltage within voltage band of +-2% p.U.

In [19]:
net = grids["four_bus_system_controlling_bus2_with_asymmetric_sgen_locked_reverse_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=2, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_reverse')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[2]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.0205298737331732, 1.0289850336961535, 1.0252874436236765]

--- Post control results ---
Check passed: True
Tap settings: [-1, -2, -1]
Voltages at controlled bus: [1.0144027907099196, 1.016542104793007, 1.0191477237188993]
Max overall vm_pu difference: 3.11316772944048e-07
Max overall va_degree difference: 0.0006421676576877644
Max overall i_from_ka difference: 2.069905649992876e-06
Max overall i_to_ka difference: 2.069905649992876e-06
Max overall p_from_mw difference: 2.03607698887609e-05
Max overall p_to_mw difference: 2.1037647361943357e-05
Max overall q_from_mvar difference: 9.009359759004631e-06
Max overall q_to_mvar difference: 7.918651429461576e-06

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000     24.720114      24.720759
      1      1.0000

## Test 15: Asymmetrical sgen at Bus3, LDC controlling Bus3 in locked_forward mode
Controller to stay inactive due to operation mode

In [20]:
net = grids["four_bus_system_controlling_bus3_with_asymmetric_sgen_locked_forward_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=3, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_forward')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.0422071770731156, 1.0625423071460705, 1.0538055284928072]

--- Post control results ---
Check passed: True
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.0422071770731156, 1.0625423071460705, 1.0538055284928072]
Max overall vm_pu difference: 4.035416608511033e-07
Max overall va_degree difference: 0.0007916724957652832
Max overall i_from_ka difference: 2.521123406817871e-06
Max overall i_to_ka difference: 2.521123406817871e-06
Max overall p_from_mw difference: 2.4955604344789606e-05
Max overall p_to_mw difference: 2.577318307306342e-05
Max overall q_from_mvar difference: 1.1478913698681481e-05
Max overall q_to_mvar difference: 1.0240014758577942e-05

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000     24.500211       0.000000
      1      1.00

## Test 16: Asymmetrical sgen at Bus3, LDC controlling Bus3 in locked_reverse mode
Controller to stabilize controlled bus voltage within voltage band of +-2% p.U.

In [21]:
net = grids["four_bus_system_controlling_bus3_with_asymmetric_sgen_locked_reverse_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=3, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_reverse')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[3]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.0422071770731156, 1.0625423071460705, 1.0538055284928072]

--- Post control results ---
Check passed: True
Tap settings: [-4, -7, -6]
Voltages at controlled bus: [1.018395902800693, 1.0198595787088947, 1.017600791586049]
Max overall vm_pu difference: 1.67455554783702e-07
Max overall va_degree difference: 0.0003529646001538822
Max overall i_from_ka difference: 1.1874898638852827e-06
Max overall i_to_ka difference: 1.1874898638852827e-06
Max overall p_from_mw difference: 1.1315361187591577e-05
Max overall p_to_mw difference: 1.1718969779472843e-05
Max overall q_from_mvar difference: 4.765429818531786e-06
Max overall q_to_mvar difference: 4.032664087993965e-06

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000     25.201661      25.202018
      1      1.0

# Tests with 20-bus system with LDC controlled transformer between external grid and low voltage level

## Test 17: LDC controlling Bus1 (index 18) in locked_forward mode
Controller to stabilize controlled bus voltage within voltage band of +-2% p.U.

In [22]:
net = grids["twenty_bus_system_controlling_bus1_locked_forward_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[18]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=18, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_forward')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[18]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9039928981492407, 0.9080424065398546, 0.9415129174802053]

--- Post control results ---
Check passed: True
Tap settings: [12, 11, 9]
Voltages at controlled bus: [0.9863298244001576, 0.9842552017893671, 1.0027694464594255]
Max overall vm_pu difference: 1.687301985908718e-07
Max overall va_degree difference: 1.2691795618913204e-05
Max overall i_from_ka difference: 3.072546701787804e-07
Max overall i_to_ka difference: 3.07254901660281e-07
Max overall p_from_mw difference: 7.229363386401788e-08
Max overall p_to_mw difference: 6.931068834148402e-08
Max overall q_from_mvar difference: 3.721112467464094e-08
Max overall q_to_mvar difference: 3.3532410235667665e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000      1.813808       1.813785
      1      1.00

## Test 18: LDC controlling Bus1 (index 18) in locked_reverse mode
Controller to stay inactive due to operation mode

In [23]:
net = grids["twenty_bus_system_controlling_bus1_locked_reverse_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[18]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=18, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_reverse')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[18]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9039928981492407, 0.9080424065398546, 0.9415129174802053]

--- Post control results ---
Check passed: True
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9039928981492407, 0.9080424065398546, 0.9415129174802053]
Max overall vm_pu difference: 5.47925851135389e-07
Max overall va_degree difference: 3.7251867241572256e-05
Max overall i_from_ka difference: 8.213261176170938e-07
Max overall i_to_ka difference: 8.213266908252415e-07
Max overall p_from_mw difference: 1.758312782665361e-07
Max overall p_to_mw difference: 1.672557865331492e-07
Max overall q_from_mvar difference: 9.656869854463013e-08
Max overall q_to_mvar difference: 8.585425565268978e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000      3.850174       0.000000
      1      1.00000

## Test 19: LDC controlling Bus2 (index 12) in locked_forward mode
Controller to stabilize controlled bus voltage within voltage band of +-2% p.U.

In [24]:
net = grids["twenty_bus_system_controlling_bus2_locked_forward_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[12]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=12, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_forward')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[12]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9615971476326665, 0.9600534322498908, 0.9537360352046754]

--- Post control results ---
Check passed: True
Tap settings: [7, 7, 5]
Voltages at controlled bus: [1.0070759489737577, 1.0055531515020544, 0.9867253505457629]
Max overall vm_pu difference: 2.7741803043657143e-07
Max overall va_degree difference: 1.9459369826613226e-05
Max overall i_from_ka difference: 4.53530506094868e-07
Max overall i_to_ka difference: 4.5353084135446586e-07
Max overall p_from_mw difference: 1.0252598454246753e-07
Max overall p_to_mw difference: 9.799587667030929e-08
Max overall q_from_mvar difference: 5.4168959711872744e-08
Max overall q_to_mvar difference: 4.85603368922255e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000      1.977934       1.977920
      1      1.00

## Test 20: LDC controlling Bus2 (index 12) in locked_reverse mode
Controller to stay inactive due to operation mode

In [25]:
net = grids["twenty_bus_system_controlling_bus2_locked_reverse_LDC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[12]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ldc_stable_state(net, controlled_bus_index=12, controlled_trafo_index=0, voltage_bandwidth=[0.98,1.02], mode='locked_reverse')
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[12]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9615971476326665, 0.9600534322498908, 0.9537360352046754]

--- Post control results ---
Check passed: True
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9615971476326665, 0.9600534322498908, 0.9537360352046754]
Max overall vm_pu difference: 5.47925851135389e-07
Max overall va_degree difference: 3.7251867241572256e-05
Max overall i_from_ka difference: 8.213261176170938e-07
Max overall i_to_ka difference: 8.213266908252415e-07
Max overall p_from_mw difference: 1.758312782665361e-07
Max overall p_to_mw difference: 1.672557865331492e-07
Max overall q_from_mvar difference: 9.656869854463013e-08
Max overall q_to_mvar difference: 8.585425565268978e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000      3.850174       0.000000
      1      1.00000